# py123d → FiftyOne: Multi-Dataset Viewer

Downloads ~100 MB of data from two fully public autonomous driving datasets — **Argoverse 2** (AWS S3, no account) and **PandaSet** (HuggingFace, free account required) — converts them via py123d's unified Arrow format, and loads both into a single FiftyOne dataset tagged by source.

| Dataset | Source | Auth | Approx. size |
|---------|--------|------|--------------|
| Argoverse 2 | AWS S3 (public) | None | ~50 MB (20 iterations) |
| PandaSet | HuggingFace mirror | Free HF account | ~50 MB (1 log) |
| **Total** | | | **~100 MB** |

---

## ⚠️ Setup Notes, Tips and Tricks

- If pyenv intercepts `pip`/`python`, use the venv binaries directly: `~/py123d-fo-env/bin/pip`
- The concrete py123d API class is `ArrowSceneAPI` — the base `SceneAPI` is abstract and cannot be instantiated
- There is no `iter_frames()` method — iteration is index-based via `get_*_at_iteration(i, id)`
- Box labels must be cast with `str().split('.')[-1]` — they are enums e.g. `AV2SensorBoxDetectionLabel.BICYCLIST`
- `np.ptp()` must be used instead of `arr.ptp()` (removed in NumPy 2.0)
- Use `matplotlib.colormaps["jet"]` instead of `plt.get_cmap("jet")` (deprecated in Matplotlib 3.7)
- Box coordinates are in world frame and must be transformed to ego frame to align with the point cloud

## Step 1: Create the virtual environment and place this notebook inside it

Run all of these in your **terminal**. The notebook is moved into the venv directory so Jupyter always finds it at a known path.

```bash
# Create the venv
python -m venv ~/py123d-fo-env
source ~/py123d-fo-env/bin/activate

# Use venv binaries directly to avoid pyenv shim interference
~/py123d-fo-env/bin/pip install --upgrade pip
~/py123d-fo-env/bin/pip install "py123d[av2,pandaset,hf]" fiftyone open3d numpy opencv-python matplotlib jupyter ipykernel

# Register as a Jupyter kernel
~/py123d-fo-env/bin/python -m ipykernel install --user --name py123d-fo-env --display-name "py123d-fo-env"

# Move this notebook into the venv directory
mv ~/Downloads/py123d_to_fiftyone.ipynb ~/py123d-fo-env/

# Launch Jupyter pointing directly at the notebook
~/py123d-fo-env/bin/jupyter notebook ~/py123d-fo-env/py123d_to_fiftyone.ipynb
```

When Jupyter opens, select **Kernel → Change Kernel → py123d-fo-env**.

## Step 2: Verify the environment

In [1]:
import sys
print(sys.executable)  # must contain py123d-fo-env, not .pyenv

import py123d
print("py123d:", py123d.__file__)  # must contain py123d-fo-env

import fiftyone as fo
import open3d as o3d
import cv2
import numpy as np
import matplotlib
import huggingface_hub
print("All imports OK")

/Users/jimmyguerrero/py123d-fo-env/bin/python
py123d: /Users/jimmyguerrero/py123d-fo-env/lib/python3.10/site-packages/py123d/__init__.py
All imports OK


## Step 3: Log in to HuggingFace

A free account is required to avoid anonymous download throttling for PandaSet.

1. Sign up at [huggingface.co](https://huggingface.co) if you don't have an account
2. Generate a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) — a **read** token is sufficient
3. Paste it when prompted below

In [2]:
from huggingface_hub import login, whoami

# Prompts for your token interactively — token is not stored in the notebook
login()

user = whoami()
print(f"Logged in as: {user['name']}")

Logged in as: jguerrero


## Step 4: Set up the data directory

In [3]:
import os

data_root = os.path.expanduser("~/py123d_data")
os.makedirs(data_root, exist_ok=True)
os.environ["PY123D_DATA_ROOT"] = data_root

print(f"PY123D_DATA_ROOT = {data_root}")
print(f"Writable: {os.access(data_root, os.W_OK)}")

PY123D_DATA_ROOT = /Users/jimmyguerrero/py123d_data
Writable: True


## Step 5: Download datasets (~100 MB total)

### 5a: Argoverse 2 — 1 log (~50 MB)

Streams from the public AWS S3 bucket, converts to Arrow, then cleans up the raw files. No account needed.

In [4]:
# No inline # comments — causes zsh parsing errors if run in terminal
!py123d-conversion \
    dataset=av2-sensor-stream \
    dataset.parser.splits='[av2-sensor_val]' \
    dataset.parser.downloader.num_logs=1

2026-06-05 08:37:23,811 INFO {/Users/jimmyguerrero/py123d-fo-env/lib/python3.10/site-packages/py123d/parser/av2/av2_sensor_parser.py:117}  AV2 Sensor streaming temp dir: /var/folders/pb/0851v_4d1rb2rkpl_l5xb_t00000gn/T/py123d-av2-sensor-k2i72h8u
2026-06-05 08:37:24,583 INFO {/Users/jimmyguerrero/py123d-fo-env/lib/python3.10/site-packages/py123d/parser/av2/av2_download.py:321}  Selected 1 / 150 logs for split=av2-sensor_val
2026-06-05 08:37:25,824 INFO {/Users/jimmyguerrero/py123d-fo-env/lib/python3.10/site-packages/py123d/parser/av2/av2_download.py:333}  AV2 sensor target directory: /var/folders/pb/0851v_4d1rb2rkpl_l5xb_t00000gn/T/py123d-av2-sensor-k2i72h8u
2026-06-05 08:37:25,824 INFO {/Users/jimmyguerrero/py123d-fo-env/lib/python3.10/site-packages/py123d/parser/av2/av2_download.py:334}  AV2 sensor bucket:           s3://argoverse/datasets/av2/sensor/
2026-06-05 08:37:25,824 INFO {/Users/jimmyguerrero/py123d-fo-env/lib/python3.10/site-packages/py123d/parser/av2/av2_download.py:335}  A

In [5]:
av2_log_dir = os.path.join(data_root, "logs", "av2-sensor_val")
av2_log_ids = os.listdir(av2_log_dir)
print(f"AV2 logs: {av2_log_ids}")
assert len(av2_log_ids) > 0, "AV2 conversion failed — check output above"

AV2 logs: ['02678d04-cc9f-3148-9f95-1ba66347dff9']


### 5b: PandaSet — 1 log via HuggingFace mirror (~50 MB)

Uses your HF token from Step 3 to avoid throttling.

In [6]:
!py123d-conversion \
    dataset=pandaset-stream \
    dataset.parser.downloader.num_logs=1

2026-06-05 08:41:01,114 INFO {/Users/jimmyguerrero/py123d-fo-env/lib/python3.10/site-packages/py123d/parser/pandaset/pandaset_parser.py:160}  PandaSet streaming: 1 logs selected by downloader
Maps PandasetParser: 0it [00:00, ?it/s]
Logs PandasetParser: 100%|███████████████████████| 1/1 [00:21<00:00, 21.90s/it]


In [7]:
# PandaSet logs land in a different split directory — find it
panda_base = os.path.join(data_root, "logs")
panda_splits = [d for d in os.listdir(panda_base) if "pandaset" in d.lower()]
print(f"PandaSet split dirs: {panda_splits}")

panda_log_dir = os.path.join(panda_base, panda_splits[0])
panda_log_ids = os.listdir(panda_log_dir)
print(f"PandaSet logs: {panda_log_ids}")
assert len(panda_log_ids) > 0, "PandaSet conversion failed — check output above"

PandaSet split dirs: ['pandaset_train']
PandaSet logs: ['001']


In [8]:
!du -sh {data_root}

732M	/Users/jimmyguerrero/py123d_data


## Step 6: Helper functions

In [9]:
JET = matplotlib.colormaps["jet"]  # colormaps[] not get_cmap() — deprecated in Matplotlib 3.7

def colorize_by_height(xyz):
    """Color LiDAR points by Z height: blue=low, red=high."""
    z = xyz[:, 2]
    z_norm = (z - z.min()) / (np.ptp(z) + 1e-6)  # np.ptp() — arr.ptp() removed in NumPy 2.0
    return JET(z_norm)[:, :3]  # RGBA -> RGB

def save_image(image_array, path):
    """Save a numpy RGB image to disk as JPEG."""
    cv2.imwrite(path, cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR))

def save_pcd(xyz, path):
    """Save a point cloud with height-based coloring to disk as PCD."""
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    pcd.colors = o3d.utility.Vector3dVector(colorize_by_height(xyz))
    o3d.io.write_point_cloud(path, pcd)

def make_3d_detections_ego(boxes, ego):
    """
    Transform box centers from world frame into ego (vehicle) frame.
    Boxes must be in ego frame to align with the point cloud in FiftyOne's 3D viewer.
    Notes:
      - ego.center_se3.inverse is a property, not a method — no () needed
      - label is an enum e.g. AV2SensorBoxDetectionLabel.BICYCLIST — split on . to get clean name
    """
    ego_inv_mat = ego.center_se3.inverse.transformation_matrix  # 4x4
    detections = []
    for box in boxes:
        world_xyz = np.array([box.center_se3.x, box.center_se3.y, box.center_se3.z, 1.0])
        ego_xyz = ego_inv_mat @ world_xyz
        rel_yaw = box.center_se3.yaw - ego.center_se3.yaw
        detections.append(fo.Detection(
            label=str(box.attributes.label).split(".")[-1],
            location=[float(ego_xyz[0]), float(ego_xyz[1]), float(ego_xyz[2])],
            dimensions=[box.bounding_box_se3.length,
                        box.bounding_box_se3.width,
                        box.bounding_box_se3.height],
            rotation=[box.center_se3.roll,
                      box.center_se3.pitch,
                      float(rel_yaw)],
        ))
    return fo.Detections(detections=detections)

print("Helpers defined")

Helpers defined


## Step 7: Load samples from a log

Reusable function that works for both AV2 and PandaSet since py123d normalises them to the same `ArrowSceneAPI` interface. Each iteration becomes a **group** with one slice per camera plus one LiDAR slice with ego-relative 3D bounding boxes.

In [10]:
from py123d.api.scene.arrow.arrow_scene_api import ArrowSceneAPI

PCD_TEMP = "/tmp/py123d_fo_pcd"
IMG_TEMP = "/tmp/py123d_fo_img"
os.makedirs(PCD_TEMP, exist_ok=True)
os.makedirs(IMG_TEMP, exist_ok=True)

def load_log_samples(log_dir, log_id, source_tag, max_iterations=20):
    """
    Load up to max_iterations frames from a single py123d log into FiftyOne samples.
    source_tag is stored on each sample so you can filter by dataset in the UI.
    Boxes are transformed from world frame into ego frame so they align with the point cloud.
    """
    scene = ArrowSceneAPI(os.path.join(log_dir, log_id))
    n = min(max_iterations, scene.number_of_iterations)
    print(f"  {source_tag} / {log_id}: {scene.number_of_iterations} iterations available, loading {n}")
    print(f"  Cameras: {scene.available_camera_names}")
    print(f"  LiDAR:   {scene.available_lidar_names}")

    # Use lidar_merged if available, otherwise fall back to the first lidar
    lidar_id = scene.available_lidar_ids[
        scene.available_lidar_names.index("lidar_merged")
        if "lidar_merged" in scene.available_lidar_names
        else 0
    ]

    samples = []
    for i in range(n):
        group = fo.Group()

        ego = scene.get_ego_state_se3_at_iteration(i)
        box_data = scene.get_box_detections_se3_at_iteration(i)
        boxes = box_data.box_detections or []

        # Camera slices
        for cam_id, cam_name in zip(scene.available_camera_ids, scene.available_camera_names):
            cam = scene.get_camera_at_iteration(i, cam_id)
            img_path = os.path.join(IMG_TEMP, f"{source_tag}_{log_id}_{cam_name}_{i:04d}.jpg")
            save_image(cam.image, img_path)
            samples.append(fo.Sample(
                filepath=img_path,
                group=group.element(cam_name),
                source=source_tag,
                log_id=log_id,
            ))

        # LiDAR slice with ego-relative 3D bounding boxes
        lidar = scene.get_lidar_at_iteration(i, lidar_id)
        pcd_path = os.path.join(PCD_TEMP, f"{source_tag}_{log_id}_{i:04d}.pcd")
        save_pcd(lidar.xyz, pcd_path)

        samples.append(fo.Sample(
            filepath=pcd_path,
            group=group.element("lidar"),
            ground_truth=make_3d_detections_ego(boxes, ego),
            source=source_tag,
            log_id=log_id,
        ))

    print(f"  → {len(samples)} samples built")
    return samples

print("load_log_samples defined")

load_log_samples defined


## Step 8: Build the combined dataset

In [11]:
# Use the first camera name from AV2 as the default group slice
_scene_tmp = ArrowSceneAPI(os.path.join(av2_log_dir, av2_log_ids[0]))
default_camera = _scene_tmp.available_camera_names[0]
print(f"Default group slice: {default_camera}")

dataset = fo.Dataset("py123d_combined", overwrite=True)
dataset.add_group_field("group", default=default_camera)

all_samples = []

# --- Argoverse 2 ---
print("\nLoading AV2...")
for log_id in av2_log_ids:
    all_samples += load_log_samples(av2_log_dir, log_id, source_tag="av2", max_iterations=20)

# --- PandaSet ---
print("\nLoading PandaSet...")
for log_id in panda_log_ids:
    all_samples += load_log_samples(panda_log_dir, log_id, source_tag="pandaset", max_iterations=20)

dataset.add_samples(all_samples)
dataset.save()
print(f"\nTotal samples in dataset: {len(dataset)}")

Default group slice: ring_front_center

Loading AV2...
  av2 / 02678d04-cc9f-3148-9f95-1ba66347dff9: 157 iterations available, loading 20
  Cameras: ['ring_front_center', 'ring_front_left', 'ring_side_left', 'ring_rear_left', 'ring_front_right', 'ring_side_right', 'ring_rear_right', 'stereo_front_left', 'stereo_front_right']
  LiDAR:   ['up_lidar', 'down_lidar', 'lidar_merged']
  → 200 samples built

Loading PandaSet...
  pandaset / 001: 80 iterations available, loading 20
  Cameras: ['back_camera', 'front_camera', 'front_left_camera', 'left_camera', 'front_right_camera', 'right_camera']
  LiDAR:   ['main_pandar64', 'front_gt', 'lidar_merged']
  → 140 samples built
 100% |█████████████████| 340/340 [2.0s elapsed, 0s remaining, 173.0 samples/s]         

Total samples in dataset: 20


## Step 9: Launch FiftyOne

Open the printed URL in your browser.

**Navigation tips:**
- Switch camera/LiDAR slices in the **Group** panel on the right
- Press **⌘+1** for top-down LiDAR view, **⌘+3** for bottom view
- Toggle `ground_truth` in the Labels panel to show/hide 3D bounding boxes
- LiDAR points are colored by height: **blue = low, red = high**
- To see ground truth labels, switch the group slice to **lidar** in the Group panel
- Filter by the `source` field to view AV2 or PandaSet separately

In [13]:
session = fo.launch_app(dataset)
print(f"FiftyOne running at: {session.url}")



███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.16.0

🤖  Ask the Docs Agent       https://docs.voxel51.com
🚀  Getting Started Guides   https://docs.voxel51.com/getting_started
💬  Join the Community       https://community.voxel51.com
🎓  Book a Workshop          https://voxel51.com/workshops

FiftyOne running at: http://localhost:5151/


## Step 10: Inspect and filter the dataset

In [ ]:
print(dataset)
print(f"\nSample fields:")
print(dataset.get_field_schema())

In [ ]:
# Sample count per source
for source in ["av2", "pandaset"]:
    view = dataset.match(fo.ViewField("source") == source)
    print(f"{source}: {len(view)} samples")

In [ ]:
# View only AV2 in the app
session.view = dataset.match(fo.ViewField("source") == "av2")

In [ ]:
# View only PandaSet in the app
session.view = dataset.match(fo.ViewField("source") == "pandaset")

In [ ]:
# Label distribution across both datasets (LiDAR slice only)
from collections import Counter

dataset.group_slice = "lidar"
label_counts = Counter(
    det.label
    for s in dataset
    if s.ground_truth is not None
    for det in s.ground_truth.detections
)
print("Label distribution:")
for label, count in label_counts.most_common():
    print(f"  {label}: {count}")

In [ ]:
# Reset to full combined view
# Note: session.view must be None or a DatasetView — cannot assign a Dataset directly
session.view = None

# Close when done
# session.close()